In [2]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')
os.environ["EXCHANGE_RATE_API_KEY"] = userdata.get('EXCHANGE_RATE_API_KEY')
# Add both as secrets in Colab's key icon (left sidebar) before running this.

In [3]:
!pip install -q langchain-huggingface langchain-core langchain-classic requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 MB/s eta 0:00:00


In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [6]:
print(multiply.invoke({"a":4,"b":5}))

20


In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [8]:
multiply.name

'multiply'

In [9]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

# tool binding

In [10]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    provider="fireworks-ai",   # tool calling needs a provider that supports it -- cerebras is unreliable here, same as structured output
    task="text-generation"
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [11]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 73, 'total_tokens': 112}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fff32-9741-7113-b7a4-01e8891a6d50-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 39, 'total_tokens': 112})

In [12]:
llm_with_tools = llm.bind_tools([multiply])
# .bind_tools() doesn't give the model the ability to RUN the tool -- it gives the
# model the tool's name/description/schema so it can DECIDE to request a call.
# Execution is still your job (see cell 19).

In [13]:
llm_with_tools.invoke('Hi how are you')
# no tool call here -- the model only reaches for a tool when the question calls for it

AIMessage(content="Hello! I'm doing great, thanks for asking. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 136, 'total_tokens': 184}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fff33-0143-7131-aea8-eca6401ba909-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 48, 'total_tokens': 184})

In [14]:
query = HumanMessage('can you multiply 3 with 1000')

In [15]:
messages = [query]

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [21]:
result=llm_with_tools.invoke(messages)

In [22]:
print(result)

content='' additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219}


In [23]:
messages.append(result)

In [24]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219})]

In [26]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 1000},
  'id': 'call_sBvHolotUdq97TO0eShqhEEf',
  'type': 'tool_call'}]

In [27]:
tool_result = multiply.invoke(result.tool_calls[0])
# .invoke() on a tool_call dict runs the actual Python function with the model's
# chosen arguments, and wraps the result as a ToolMessage

In [28]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_sBvHolotUdq97TO0eShqhEEf')

In [29]:
messages.append(tool_result)

In [30]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_sBvHolotUdq97TO0eShqhEEf')]

In [31]:
llm_with_tools.invoke(messages).content
# THE FULL LOOP: ask -> model requests a tool call -> you run it -> feed result back
# -> model uses that result to answer in natural language

'3\u202f×\u202f1000\u202f=\u202f3000.'

##Multi-tool example (currency conversion)

In [32]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  api_key = os.environ["EXCHANGE_RATE_API_KEY"]   # your OWN key, not hardcoded
  url = f'https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

Concept: InjectedToolArg marks conversion_rate as an argument the LLM should never fill in itself — it's meant to be injected by your code (from the first tool's output), not guessed by the model. This is how you chain tool outputs safely.

In [33]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [34]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'PKR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786665602,
 'time_last_update_utc': 'Fri, 14 Aug 2026 00:00:02 +0000',
 'time_next_update_unix': 1786752002,
 'time_next_update_utc': 'Sat, 15 Aug 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'PKR',
 'conversion_rate': 277.8944}

In [35]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 277.8944})

2778.9440000000004

In [36]:
# tool binding
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    provider="fireworks-ai",
    task="text-generation"
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [37]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [38]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_sBvHolotUdq97TO0eShqhEEf')]

In [39]:
ai_message = llm_with_tools.invoke(messages)

In [40]:
messages.append(ai_message)

In [41]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_sBvHolotUdq97TO0eShqhEEf'),
 AIMessage(content='Sure! \\(3 \\times 1000 = 3000\\).', additional_kwargs={}, response_meta

In [42]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate   # THIS is where InjectedToolArg gets filled — by YOUR code, not the LLM
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [43]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{\n"a": 3,\n"b": 1000\n}', 'name': 'multiply', 'description': None}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 141, 'total_tokens': 219}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff34-cb88-7760-818a-8cb73de21143-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_sBvHolotUdq97TO0eShqhEEf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 78, 'total_tokens': 219}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_sBvHolotUdq97TO0eShqhEEf'),
 AIMessage(content='Sure! \\(3 \\times 1000 = 3000\\).', additional_kwargs={}, response_meta

In [44]:
llm_with_tools.invoke(messages).content

'Sure! \\(3 \\times 1000 = 3000\\).'

###Full agent (automates the manual loop above)


In [45]:
from langchain_classic.agents import initialize_agent, AgentType   # was langchain.agents

agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # ReAct pattern
    verbose=True  # shows internal thinking
)

/tmp/ipykernel_1206/3838980633.py:3: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent_executor = initialize_agent(


In [46]:
user_query = "Hi how are you?"
response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
{
  "action": "Final Answer",
  "action_input": "Hello! I'm doing well, thank you for asking. How can I assist you today?"
}

> Finished chain.


In [47]:
response = agent_executor.invoke({"input": "What is the conversion factor between PKR and USD, and convert 10 pkr to usd"})
print(response['output'])



> Entering new AgentExecutor chain...
{
  "action": "get_conversion_factor",
  "action_input": {
    "base_currency": "PKR",
    "target_currency": "USD"
  }
}

> Finished chain.
{
  "action": "get_conversion_factor",
  "action_input": {
    "base_currency": "PKR",
    "target_currency": "USD"
  }
}
